In [ ]:
##################################################################
# # ! Two methods to approximate a sphere with a polyhedron:
# (1) Recursive subdivision of an icosahedron
# (2) Triangulated regular geographic grid
##################################################################
# %%
# # ! Setup
import pyvista as pv
import numpy as np
from gravity_forward_numba import *
from gravity_forward_numpy import *

In [ ]:
# # ! Recursive subdivision of an icosahedron
nsub_max = 9;
Nsubs = np.arange(nsub_max);
NVs = np.zeros(nsub_max, dtype=int);
NFs = np.zeros(nsub_max, dtype=int);
Es_vol   = np.zeros(nsub_max);
Es_area  = np.zeros(nsub_max);
MINs_psi = np.zeros(nsub_max);
MAXs_psi = np.zeros(nsub_max);
vol = 4*np.pi/3; area = 4*np.pi;
for i, nsub in enumerate(Nsubs):
    icosph = pv.Icosphere(nsub=nsub);
    Verts = icosph.points;
    Faces = icosph.regular_faces;
    NVs[i] = Verts.shape[0];
    NFs[i] = Faces.shape[0];
    Es_vol[i]  = 1. - icosph.volume/vol;
    Es_area[i] = 1. - icosph.area/area;
    min_psi, max_psi = spherical_edge_length_range(Verts, Faces);
    MINs_psi[i] = 60. * np.rad2deg(min_psi); # in arc-min
    MAXs_psi[i] = 60. * np.rad2deg(max_psi);
print(f"{'N':>4} {'Nv':>8} {'Nf':>8} "
      f"{'Vol_err':>10} {'Area_err':>10} {'Min_psi':>10} {'Max_psi':>10}");
print("-" * 68);
for i in range(len(Nsubs)):
    print(f"{Nsubs[i]:>4} {int(NVs[i]):>8} {int(NFs[i]):>8}"
          f"{Es_vol[i]:>10.2e} {Es_area[i]:>10.2e}"
          f"{MINs_psi[i]:>10.3f} {MAXs_psi[i]:>10.3f}");

   N       Nv       Nf    Vol_err   Area_err    Min_psi    Max_psi
--------------------------------------------------------------------
   0       12       20  3.95e-01   2.38e-01  3806.097   3806.097
   1       42       80  1.27e-01   7.17e-02  1903.048   2160.000
   2      162      320  3.38e-02   1.89e-02   872.726   1121.964
   3      642     1280  8.62e-03   4.79e-03   410.913    566.657
   4     2562     5120  2.17e-03   1.20e-03   198.831    284.052
   5    10242    20480  5.42e-04   3.01e-04    97.751    142.117
   6    40962    81920  1.36e-04   7.52e-05    48.459     71.070
   7   163842   327680  3.39e-05   1.88e-05    24.126     35.536
   8   655362  1310720  8.47e-06   4.70e-06    12.037     17.768


In [ ]:
# # ! Triangulated regular geographic grid
Lon_Res = 6*2**np.arange(nsub_max)+1;
Lat_Res = 3*2**np.arange(nsub_max)+1;
for i, (lon_res, lat_res) in enumerate(zip(Lon_Res, Lat_Res)):
    regsph = pv.Sphere(radius=1.0, 
                       theta_resolution=lat_res, phi_resolution=lon_res);
    Verts = regsph.points;
    Faces = regsph.regular_faces;
    NVs[i] = Verts.shape[0];
    NFs[i] = Faces.shape[0];
    Es_vol[i]  = 1. - regsph.volume/vol;
    Es_area[i] = 1. - regsph.area/area;
    min_psi, max_psi = spherical_edge_length_range(Verts, Faces);
    MINs_psi[i] = 60. * np.rad2deg(min_psi); # in arc-min
    MAXs_psi[i] = 60. * np.rad2deg(max_psi);
print(f"{'N':>4} {'Nv':>8} {'Nf':>8} "
      f"{'Vol_err':>10} {'Area_err':>10} {'Min_psi':>10} {'Max_psi':>10}");
print("-" * 68);
for i in range(len(Nsubs)):
    print(f"{Nsubs[i]:>4} {int(NVs[i]):>8} {int(NFs[i]):>8}"
          f"{Es_vol[i]:>10.2e} {Es_area[i]:>10.2e}"
          f"{MINs_psi[i]:>10.3f} {MAXs_psi[i]:>10.3f}");

   N       Nv       Nf    Vol_err   Area_err    Min_psi    Max_psi
--------------------------------------------------------------------
   0       22       40  4.06e-01   2.12e-01  1800.000   5400.000
   1       79      154  1.44e-01   7.27e-02   773.732   3178.149
   2      301      598  4.26e-02   2.14e-02   214.804   1716.734
   3     1177     2350  1.16e-02   5.78e-03    56.360    892.217
   4     4657     9310  3.01e-03   1.50e-03    14.413    454.869
   5    18529    37054  7.66e-04   3.83e-04     3.643    229.666
   6    73921   147838  1.93e-04   9.67e-05     0.916    115.396
   7   295297   590590  4.86e-05   2.43e-05     0.229     57.840
   8  1180417  2360830  1.22e-05   6.09e-06     0.057     28.955


In [ ]:
# # ! Plot
pl = pv.Plotter(shape=(2, 3), image_scale=3);
for i, nsub in enumerate([1, 2, 3]):
    icosph = pv.Icosphere(nsub=nsub);
    icosph_with_area = icosph.compute_cell_sizes();
    areas = icosph_with_area['Area'];
    area_percent = 100 * areas / areas.sum();
    scalar_name = f'Area (%), n={nsub}';
    icosph_with_area[scalar_name] = area_percent;
    pl.subplot(0, i);
    sargs = dict(title=scalar_name, title_font_size=14, label_font_size=12, 
                 n_labels=3, position_y=0.05, fmt='%.2f');
    pl.add_mesh(icosph_with_area, scalars=scalar_name, 
                scalar_bar_args=sargs, cmap='viridis'); # viridis, plasma, magma, turbo
    pl.camera.Zoom(1.25);
pl.show();
pl.screenshot("icosphere_geographic");

Widget(value='<iframe src="http://localhost:43417/index.html?ui=P_0x726a42792190_0&reconnect=auto" class="pyvi…